In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from matplotlib import pyplot as plt
from keras.models import Sequential
import kagglehub

In [2]:
import os
from google.colab import userdata

os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

import kagglehub
path = kagglehub.dataset_download("prasunroy/natural-images")
print(path)

Using Colab cache for faster access to the 'natural-images' dataset.
/kaggle/input/natural-images


In [3]:
os.listdir(path)

['natural_images', 'data']

In [4]:
print(os.listdir(os.path.join(path, "natural_images")))
print(os.listdir(os.path.join(path, "data")))

['motorbike', 'airplane', 'flower', 'dog', 'fruit', 'car', 'cat', 'person']
['natural_images']


In [5]:
data_dir = os.path.join(path, "natural_images")

In [6]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    directory=data_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(160, 160),
    batch_size=64
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    directory=data_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(160, 160),
    batch_size=64
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

Found 6899 files belonging to 8 classes.
Using 5520 files for training.
Found 6899 files belonging to 8 classes.
Using 1379 files for validation.


In [7]:
model_cnn = Sequential([
    keras.layers.Rescaling(1./255),
    keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(160, 160, 3)),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D((2, 2)),

    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D((2, 2)),

    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D((2, 2)),

    keras.layers.Conv2D(128, (3, 3), activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Flatten(),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(128, activation='relu'),

    keras.layers.Dense(8, activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [8]:
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=0.00001
)

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [9]:
model_cnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [10]:
history_cnn = model_cnn.fit(train_ds, epochs=50, validation_data=val_ds, callbacks=[reduce_lr, early_stop])

Epoch 1/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 28s 191ms/step - accuracy: 0.7978 - loss: 1.3157 - val_accuracy: 0.1255 - val_loss: 17.6315 - learning_rate: 0.0010
Epoch 2/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 21s 57ms/step - accuracy: 0.8902 - loss: 0.5751 - val_accuracy: 0.1327 - val_loss: 26.5998 - learning_rate: 0.0010
Epoch 3/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 5s 60ms/step - accuracy: 0.9304 - loss: 0.2782 - val_accuracy: 0.2451 - val_loss: 14.8022 - learning_rate: 0.0010
Epoch 4/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 5s 58ms/step - accuracy: 0.9476 - loss: 0.2312 - val_accuracy: 0.4953 - val_loss: 4.3226 - learning_rate: 0.0010
Epoch 5/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 5s 57ms/step - accuracy: 0.9667 - loss: 0.1167 - val_accuracy: 0.6759 - val_loss: 4.6313 - learning_rate: 0.0010
Epoch 6/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 5s 58ms/step - accuracy: 0.9750 - loss: 0.0862 - val_accuracy: 0.8318 - val_loss: 0.9726 - learning_rate: 0.0010
Epoch 7/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 5s 59ms/step - accuracy: 0.9875 - loss: 0.0382 - v

In [11]:
from keras.applications.vgg16 import preprocess_input, VGG16

conv_base = VGG16(weights="imagenet", include_top=False, input_shape=(160, 160, 3))
conv_base.trainable = False

model_vgg = Sequential([
    conv_base,
    keras.layers.Flatten(),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(8, activation='softmax')
])

In [12]:
model_vgg.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [13]:
history_vgg = model_vgg.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds
)

Epoch 1/10
87/87 ━━━━━━━━━━━━━━━━━━━━ 58s 472ms/step - accuracy: 0.9520 - loss: 2.0284 - val_accuracy: 0.9913 - val_loss: 0.3145
Epoch 2/10
87/87 ━━━━━━━━━━━━━━━━━━━━ 23s 268ms/step - accuracy: 0.9862 - loss: 0.3862 - val_accuracy: 0.9877 - val_loss: 0.4692
Epoch 3/10
87/87 ━━━━━━━━━━━━━━━━━━━━ 23s 266ms/step - accuracy: 0.9908 - loss: 0.2855 - val_accuracy: 0.9906 - val_loss: 0.4845
Epoch 4/10
87/87 ━━━━━━━━━━━━━━━━━━━━ 23s 265ms/step - accuracy: 0.9924 - loss: 0.2406 - val_accuracy: 0.9906 - val_loss: 0.5423
Epoch 5/10
87/87 ━━━━━━━━━━━━━━━━━━━━ 23s 266ms/step - accuracy: 0.9928 - loss: 0.2592 - val_accuracy: 0.9906 - val_loss: 0.4998
Epoch 6/10
87/87 ━━━━━━━━━━━━━━━━━━━━ 23s 266ms/step - accuracy: 0.9958 - loss: 0.1233 - val_accuracy: 0.9891 - val_loss: 0.5718
Epoch 7/10
87/87 ━━━━━━━━━━━━━━━━━━━━ 23s 267ms/step - accuracy: 0.9933 - loss: 0.2444 - val_accuracy: 0.9898 - val_loss: 0.7109
Epoch 8/10
87/87 ━━━━━━━━━━━━━━━━━━━━ 23s 266ms/step - accuracy: 0.9944 - loss: 0.2709 - val_accu

In [14]:
conv_base.trainable = True
for layer in conv_base.layers:
    if "block5" in layer.name:
        layer.trainable = True
    else:
        layer.trainable = False

In [15]:
model_vgg.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5), # 0.00001
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [16]:
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7
)

In [17]:
history_vgg_tuned = model_vgg.fit(
    train_ds,
    epochs=15,
    validation_data=val_ds,
    callbacks=[reduce_lr]
)

Epoch 1/15
87/87 ━━━━━━━━━━━━━━━━━━━━ 38s 353ms/step - accuracy: 0.9947 - loss: 0.2148 - val_accuracy: 0.9920 - val_loss: 0.4376 - learning_rate: 1.0000e-05
Epoch 2/15
87/87 ━━━━━━━━━━━━━━━━━━━━ 26s 303ms/step - accuracy: 0.9987 - loss: 0.0103 - val_accuracy: 0.9920 - val_loss: 0.4028 - learning_rate: 1.0000e-05
Epoch 3/15
87/87 ━━━━━━━━━━━━━━━━━━━━ 26s 300ms/step - accuracy: 0.9982 - loss: 0.0267 - val_accuracy: 0.9906 - val_loss: 0.4988 - learning_rate: 1.0000e-05
Epoch 4/15
87/87 ━━━━━━━━━━━━━━━━━━━━ 26s 298ms/step - accuracy: 0.9986 - loss: 0.0276 - val_accuracy: 0.9906 - val_loss: 0.4206 - learning_rate: 1.0000e-05
Epoch 5/15
87/87 ━━━━━━━━━━━━━━━━━━━━ 26s 301ms/step - accuracy: 0.9986 - loss: 0.0349 - val_accuracy: 0.9913 - val_loss: 0.4350 - learning_rate: 1.0000e-05
Epoch 6/15
87/87 ━━━━━━━━━━━━━━━━━━━━ 26s 303ms/step - accuracy: 1.0000 - loss: 2.3279e-08 - val_accuracy: 0.9920 - val_loss: 0.4198 - learning_rate: 5.0000e-06
Epoch 7/15
87/87 ━━━━━━━━━━━━━━━━━━━━ 26s 301ms/step -

In [18]:
from keras.applications import MobileNetV2
from keras.applications.mobilenet_v2 import preprocess_input


train_ds = tf.keras.utils.image_dataset_from_directory(
    directory=data_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(160, 160),
    batch_size=64
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    directory=data_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(160, 160),
    batch_size=64
)


def process_data(image, label):
    return preprocess_input(image), label

train_ds = train_ds.map(process_data).cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.map(process_data).cache().prefetch(buffer_size=AUTOTUNE)


conv_base_mob = MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights='imagenet'
)
conv_base_mob.trainable = False


model_mob = Sequential([
    conv_base_mob,
    keras.layers.Flatten(),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(8, activation='softmax')
])

Found 6899 files belonging to 8 classes.
Using 5520 files for training.
Found 6899 files belonging to 8 classes.
Using 1379 files for validation.


In [19]:
model_mob.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [20]:
history_mob = model_mob.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds
)

Epoch 1/10
87/87 ━━━━━━━━━━━━━━━━━━━━ 55s 417ms/step - accuracy: 0.9676 - loss: 0.2791 - val_accuracy: 0.9971 - val_loss: 0.0308
Epoch 2/10
87/87 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - accuracy: 0.9918 - loss: 0.0743 - val_accuracy: 0.9964 - val_loss: 0.0503
Epoch 3/10
87/87 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - accuracy: 0.9951 - loss: 0.0402 - val_accuracy: 0.9949 - val_loss: 0.0657
Epoch 4/10
87/87 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - accuracy: 0.9958 - loss: 0.0506 - val_accuracy: 0.9956 - val_loss: 0.0599
Epoch 5/10
87/87 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step - accuracy: 0.9946 - loss: 0.0619 - val_accuracy: 0.9927 - val_loss: 0.0721
Epoch 6/10
87/87 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - accuracy: 0.9982 - loss: 0.0168 - val_accuracy: 0.9985 - val_loss: 0.0234
Epoch 7/10
87/87 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - accuracy: 0.9975 - loss: 0.0228 - val_accuracy: 0.9956 - val_loss: 0.0585
Epoch 8/10
87/87 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - accuracy: 0.9967 - loss: 0.0658 - val_accuracy: 0.9978 -

In [21]:
conv_base_mob.trainable = True
for layer in conv_base_mob.layers:
    if "block_16" in layer.name or "Conv_1" in layer.name:
        layer.trainable = True
    else:
        layer.trainable = False

In [22]:
model_mob.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5), # 0.00001
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [23]:
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7
)

In [24]:
history_mob_tuned = model_mob.fit(
    train_ds,
    epochs=15,
    validation_data=val_ds,
    callbacks=[reduce_lr]
)

Epoch 1/15
87/87 ━━━━━━━━━━━━━━━━━━━━ 32s 213ms/step - accuracy: 0.9748 - loss: 0.5909 - val_accuracy: 0.9949 - val_loss: 0.1521 - learning_rate: 1.0000e-05
Epoch 2/15
87/87 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step - accuracy: 0.9928 - loss: 0.1513 - val_accuracy: 0.9949 - val_loss: 0.1336 - learning_rate: 1.0000e-05
Epoch 3/15
87/87 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - accuracy: 0.9944 - loss: 0.1020 - val_accuracy: 0.9942 - val_loss: 0.1229 - learning_rate: 1.0000e-05
Epoch 4/15
87/87 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - accuracy: 0.9955 - loss: 0.0513 - val_accuracy: 0.9956 - val_loss: 0.1114 - learning_rate: 1.0000e-05
Epoch 5/15
87/87 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step - accuracy: 0.9969 - loss: 0.0412 - val_accuracy: 0.9964 - val_loss: 0.1056 - learning_rate: 1.0000e-05
Epoch 6/15
87/87 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step - accuracy: 0.9982 - loss: 0.0098 - val_accuracy: 0.9964 - val_loss: 0.1023 - learning_rate: 1.0000e-05
Epoch 7/15
87/87 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - accuracy: 0.998

In [ ]:
model_cnn.save('model_cnn.keras')
model_vgg.save('model_vgg.keras')
model_mob.save('model_mob.keras')

import pickle

with open('history_cnn.pkl', 'wb') as f:
    pickle.dump(history_cnn.history, f)

with open('history_vgg.pkl', 'wb') as f:
    pickle.dump(history_vgg.history, f)

with open('history_vgg_tuned.pkl', 'wb') as f:
    pickle.dump(history_vgg_tuned.history, f)

with open('history_mob.pkl', 'wb') as f:
    pickle.dump(history_mob.history, f)

with open('history_mob_tuned.pkl', 'wb') as f:
    pickle.dump(history_mob_tuned.history, f)

print("All saved!")

In [26]:
from google.colab import files

files.download('model_cnn.keras')
files.download('model_vgg.keras')
files.download('model_mob.keras')
files.download('history_cnn.pkl')
files.download('history_vgg.pkl')
files.download('history_vgg_tuned.pkl')
files.download('history_mob.pkl')
files.download('history_mob_tuned.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>